In [1]:
import brainsss
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
import time
import h5py
import ants
import nibabel as nib
from scipy.ndimage import uniform_filter, gaussian_filter
import shutil
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
import sklearn.linear_model as lnm
from sklearn.decomposition import PCA
import gc
import sys
import warnings
from scipy.ndimage import gaussian_filter1d,gaussian_filter
from scipy.signal import butter, sosfiltfilt, filtfilt, freqz,iirnotch
import pickle
# %matplotlib notebook

/home/users/ilanazs/.local/lib/python3.6/site-packages/ants/viz/render_surface_function.py:16: UserWarning:

Cant import Plotly. Install it `pip install chart_studio` if you want to use ants.render_surface_function



In [14]:
### big STA supervox
later_dir = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/temp_filter'
cluster_dir = os.path.join(later_dir, 'clustering')
save_dir=os.path.join(later_dir,'figs')
dataset_path="/oak/stanford/groups/trc/data/Ilana/2P/data"
scratch_path="/scratch/users/ilanazs"
fly='fly_239'

fly_directory = os.path.join(dataset_path, fly)
fly=fly
later_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/'
save_directory = os.path.join(scratch_path, fly)
timestamp_file = "timestamps_warp.h5"
event = 'flat'
redo=False
cc = '2'
# behavior = args['behavior']

load_path = os.path.join(fly_directory, timestamp_file)
brain_file = f"functional_channel_{cc}_moco_warp_blurred_hpf_dff.h5"
scratch_dir=os.path.join(scratch_path, fly)

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [15]:

event_times_path = os.path.join(later_path, f'{event}_event_times_split_dic.pkl')

with open(event_times_path, 'rb') as file:
    event_times_struct = pickle.load(file)
    f=list(event_times_struct.keys())[0]
    behaviors=list(event_times_struct[f].keys())
print(f"Event times from {event_times_path}")

behavior=behaviors[0]
print(behavior)



Event times from /oak/stanford/groups/trc/data/Ilana/2P/data/later/flat_event_times_split_dic.pkl
flat


In [16]:
filter_load_path=os.path.join(scratch_dir, f"filter_needs_{cc}_{behavior}_{event}.h5")
ts_rel_load_path=os.path.join(scratch_dir, f"ts_rel_odd_mask_{cc}_{behavior}_{event}.h5")
save_file = os.path.join(save_directory, brain_file.split('.')[0] + '_filtered_' + f'{behavior}_{event}.h5')
print(f'filter path:{filter_load_path}\nts path:{ts_rel_load_path}\nsave path:{save_file}')

filter path:/scratch/users/ilanazs/fly_239/filter_needs_2_flat_flat.h5
ts path:/scratch/users/ilanazs/fly_239/ts_rel_odd_mask_2_flat_flat.h5
save path:/scratch/users/ilanazs/fly_239/functional_channel_2_moco_warp_blurred_hpf_dff_filtered_flat_flat.h5


In [17]:
load_directory=load_directory = os.path.join(fly_directory, "dff")
brain_load_path = os.path.join(load_directory, brain_file)
ts_load_path = os.path.join(fly_directory, 'warp', timestamp_file)
save_directory = os.path.join(fly_directory, "temp_filter")

In [20]:
brain_dest=os.path.join(scratch_dir, brain_file)
ts_dest=os.path.join(scratch_dir, timestamp_file)
# if os.path.exists(brain_dest)==False: 
#     b_dest = shutil.copyfile(brain_load_path, brain_dest)
# if os.path.exists(ts_dest)==False: 
#     t_dest = shutil.copyfile(ts_load_path, ts_dest)
print(f"brain in {brain_dest}\ntimestamp in {ts_dest}")

brain in /scratch/users/ilanazs/fly_239/functional_channel_2_moco_warp_blurred_hpf_dff.h5
timestamp in /scratch/users/ilanazs/fly_239/timestamps_warp.h5


In [21]:
%%time
with h5py.File(brain_dest, 'r') as hf, \
    h5py.File(ts_dest, 'r') as tf, \
    h5py.File(ts_rel_load_path, 'r') as of, \
    h5py.File(filter_load_path, 'r') as ff:

    brain_all = hf['data'][:]
    ts_all = tf['data'][:]
    loom_all = ff['loom_starts'][:]
    bin_shape = ff['bin_shape'][:]
    odd_mask = of['odd_mask'][:]
    ts_rel = of['ts_rel'][:]

CPU times: user 40 µs, sys: 1min 32s, total: 1min 32s
Wall time: 1min 32s


In [29]:
dims=np.shape(brain_all)
print(F"Brain data shape is {dims}")

T=(ts_all[0,0,0,1]-ts_all[0,0,0,0])/1000
fs=1/T #sample rate, Hz
if loom_all.shape[0]>0:
    max_len=int((((bin_shape[1]-bin_shape[0])/1000)*fs)*np.shape(loom_all)[0])+100
else:
    max_len=loom_all.shape[0]
print(F"Max length of filtered data is {max_len}")

nx, ny, nz, nt = brain_all.shape

Brain data shape is (314, 146, 91, 3384)
Max length of filtered data is 0


In [28]:
loom_all.shape[0]

0